# 📖 PDF → audiobook mp3 bằng giọng clone (OmniVoice) trên Colab

Upload **một file PDF** + **clip giọng mẫu**. Notebook tách câu, render bằng giọng clone trên GPU T4, rồi **ghép thành file mp3** (mỗi chương một file, hoặc cả cuốn một file) lưu trên Google Drive để bạn tải về.

Render được lưu **từng đợt lên Drive**: runtime chết giữa chừng thì chạy lại, nó tiếp tục từ chỗ dừng.

**Luồng:** upload → tách câu & kiểm tra bằng mắt → gán sắc thái theo ngữ cảnh → *benchmark 30 câu* → render hết → báo cáo → ghép mp3 → tải về.

## 0. Kiểm tra GPU (phải thấy Tesla T4)

In [ ]:
!nvidia-smi

## 1. Cài đặt

In [ ]:
!pip install -q omnivoice soundfile pymupdf anthropic
print('Xong.')

## 2a. Mount Drive + upload PDF

Chạy ô dưới, bấm nút và chọn **một file PDF**. Notebook copy nó vào Drive.
Runtime chết giữa chừng: điền `BOOK` đúng tên thư mục trên Drive thì ô này bỏ qua bước upload và nạp lại sách đã lưu.

In [ ]:
from google.colab import drive, files
import re, shutil, unicodedata
from pathlib import Path

drive.mount('/content/drive')

BOOK = ''   # trống = sách mới; điền tên thư mục (vd 'dac_nhan_tam') để tiếp tục sách đã upload
BASE = Path('/content/drive/MyDrive/omnivoice_render')

def slugify(name):
    s = Path(name).stem.replace('đ', 'd').replace('Đ', 'D')
    s = unicodedata.normalize('NFKD', s).encode('ascii', 'ignore').decode()
    return re.sub(r'[^a-zA-Z0-9]+', '_', s).strip('_').lower() or 'book'

REF_AUDIO = None
if BOOK and (BASE/BOOK/'source.pdf').exists():
    ROOT = BASE/BOOK
    PDF = str(ROOT/'source.pdf')
    refs = sorted(ROOT.glob('ref.*'))
    REF_AUDIO = str(refs[0]) if refs else None
    print('Tiếp tục sách:', BOOK, '| giọng mẫu đã lưu:', Path(REF_AUDIO).name if REF_AUDIO else 'chưa có')
else:
    if BOOK:
        print(f"Không thấy '{BOOK}' trên Drive → coi như sách mới.")
    up = files.upload()
    pdf_name = next((n for n in up if n.lower().endswith('.pdf')), None)
    assert pdf_name, 'Bạn chưa chọn file .pdf. Chạy lại ô này.'
    BOOK = slugify(pdf_name)
    ROOT = BASE/BOOK
    ROOT.mkdir(parents=True, exist_ok=True)
    shutil.copy(pdf_name, ROOT/'source.pdf')
    PDF = str(ROOT/'source.pdf')

print('Sách:', BOOK, '| thư mục Drive:', ROOT)

## 2b. Upload giọng mẫu

Chạy ô dưới, bấm nút và chọn **một clip giọng** (wav/mp3/m4a/flac/ogg).
Sách đã có giọng mẫu lưu trên Drive thì ô này tự bỏ qua. Muốn đổi sang giọng khác: đặt `REPLACE_VOICE = True` (và **nhớ đổi `VOICE_ID`** ở mục 5, để cache không lẫn giọng cũ).

In [ ]:
REPLACE_VOICE = False   # True = upload giọng mẫu mới cho sách này

assert 'ROOT' in globals(), 'Hãy chạy ô 2a (upload PDF) trước.'
if REF_AUDIO and not REPLACE_VOICE:
    print('Dùng giọng mẫu đã lưu:', Path(REF_AUDIO).name)
else:
    up = files.upload()
    ref_name = next((n for n in up if n.lower().endswith(('.wav', '.mp3', '.m4a', '.flac', '.ogg'))), None)
    assert ref_name, 'Bạn chưa chọn clip giọng (.wav/.mp3/.m4a/.flac/.ogg). Chạy lại ô này.'
    for old in ROOT.glob('ref.*'):
        old.unlink()
    ref_dst = ROOT/('ref' + Path(ref_name).suffix)
    shutil.copy(ref_name, ref_dst)
    REF_AUDIO = str(ref_dst)
    print('Đã lưu giọng mẫu:', ref_dst.name)

## 3a. Module chuẩn hoá văn bản trước TTS (`text_norm.py`)

Bản sao của `backend/text_norm.py` trong repo kindle-voice. Chỉ thay đổi **phần được đọc**, còn `text` nguyên văn vẫn giữ để bạn xem lại:
- tiêu đề/câu **VIẾT HOA toàn bộ** được chuyển về chữ thường (vd "TÔI ĐÃ SỐNG…" thành "Tôi đã sống…"). Nếu để nguyên, TTS đọc chữ hoa tiếng Việt như từ viết tắt hoặc tiếng nước ngoài. Từ viết tắt đứng riêng (NXB, TP.HCM) được giữ nguyên;
- số La Mã ở đầu mục được đọc thành chữ ("Chương IV" thành "Chương bốn");
- ký hiệu lạ (mũi tên, emoji…) bị bỏ để model không đọc ra âm rác.

In [ ]:
%%writefile text_norm.py
"""text_norm.py — Chuẩn hoá văn bản NGAY TRƯỚC KHI đưa vào TTS.

Đặt ở tầng phát âm (engine) chứ KHÔNG ở parser: nhờ vậy màn hình vẫn hiển thị
nguyên văn ("I", "II", "Chương III"...) trong khi phần đọc được chuẩn hoá cho
đúng và tránh model sinh âm rác.

Xử lý ba vấn đề chính:
  1) Đầu mục đánh số La Mã (I, II, "Chương IV"...) đọc thành số đếm tiếng Việt
     ("một", "hai", "Chương bốn").
  2) Tiêu đề / câu VIẾT HOA TOÀN BỘ ("TÔI ĐÃ SỐNG MỘT CUỘC ĐỜI...") chuyển về chữ
     thường: TTS đọc chữ hoa tiếng Việt như từ viết tắt / tiếng nước ngoài.
  3) Ký tự lạ / không đọc được (ký hiệu, mũi tên, emoji, dấu chú thích sót lại)
     bị loại bỏ để neural TTS không "babble" ra đoạn vô nghĩa không có trong sách.
"""
from __future__ import annotations

import re
import unicodedata
from typing import Optional

# --------------------------------------------------------------------------- #
# 1) Số La Mã -> chữ tiếng Việt                                                #
# --------------------------------------------------------------------------- #

# Số La Mã HỢP LỆ (in hoa), 1..3999. Dùng để tránh nhận nhầm chữ thường gặp.
_ROMAN_RE = re.compile(
    r"^M{0,3}(CM|CD|D?C{0,3})(XC|XL|L?X{0,3})(IX|IV|V?I{0,3})$"
)

# Từ khoá đứng trước số thứ tự trong đầu mục (sau các từ này, số La Mã chắc chắn
# là số thứ tự nên chuyển an toàn). So khớp không phân biệt hoa/thường.
_INDEX_WORDS = (
    "chương", "phần", "mục", "hồi", "bài", "tập", "quyển", "kỳ", "chuyện",
    "chapter", "part", "book", "section", "act", "canto",
)
_KEYWORD_ROMAN_RE = re.compile(
    r"\b(" + "|".join(_INDEX_WORDS) + r")\s+([IVXLCDM]+)\b",
    re.IGNORECASE,
)
# Số La Mã đứng ĐẦU chuỗi, theo sau là dấu phân cách hoặc hết chuỗi:
#   "I", "II.", "III -", "IV: Mở đầu", "(V)"...
_LEADING_ROMAN_RE = re.compile(
    r"^\s*[(\[]?\s*([IVXLCDM]+)\s*[)\]]?\s*(?=[.)\]:\-—–]|\s|$)"
)

_ONES = ["", "một", "hai", "ba", "bốn", "năm", "sáu", "bảy", "tám", "chín"]
_TENS_UNIT = {1: "mốt", 4: "tư", 5: "lăm"}  # biến âm ở hàng đơn vị khi có hàng chục


def roman_to_int(s: str) -> Optional[int]:
    """Chuyển chuỗi số La Mã (in hoa) -> int, hoặc None nếu không hợp lệ."""
    s = (s or "").upper()
    if not s or not _ROMAN_RE.match(s):
        return None
    vals = {"I": 1, "V": 5, "X": 10, "L": 50, "C": 100, "D": 500, "M": 1000}
    total, prev = 0, 0
    for ch in reversed(s):
        v = vals[ch]
        total += -v if v < prev else v
        prev = v
    return total


def _under_hundred(n: int) -> str:
    """0..99 -> chữ (dùng cho ghép hàng trăm)."""
    if n < 10:
        return _ONES[n]
    tens, unit = divmod(n, 10)
    head = "mười" if tens == 1 else f"{_ONES[tens]} mươi"
    if unit == 0:
        return head
    return f"{head} {_TENS_UNIT.get(unit, _ONES[unit])}"


def int_to_vietnamese(n: int) -> str:
    """Đọc số nguyên dương -> chữ tiếng Việt (đủ dùng cho đầu mục, 1..3999)."""
    if n <= 0:
        return str(n)
    if n < 100:
        return _under_hundred(n)
    if n < 1000:
        hundreds, rest = divmod(n, 100)
        out = f"{_ONES[hundreds]} trăm"
        if rest == 0:
            return out
        if rest < 10:                     # "linh" cho hàng chục trống
            return f"{out} linh {_ONES[rest]}"
        return f"{out} {_under_hundred(rest)}"
    thousands, rest = divmod(n, 1000)
    out = f"{int_to_vietnamese(thousands)} nghìn"
    if rest == 0:
        return out
    if rest < 100:
        return f"{out} không trăm {_under_hundred(rest)}" if rest >= 10 \
            else f"{out} không trăm linh {_ONES[rest]}"
    return f"{out} {int_to_vietnamese(rest)}"


def _roman_word(token: str) -> Optional[str]:
    n = roman_to_int(token)
    return int_to_vietnamese(n) if n else None


def _convert_roman_headings(text: str) -> str:
    """Chuyển số La Mã trong đầu mục -> chữ tiếng Việt (giữ nguyên phần còn lại)."""
    # a) "Chương IV" -> "Chương bốn"
    def _kw(m: re.Match) -> str:
        word = _roman_word(m.group(2))
        return f"{m.group(1)} {word}" if word else m.group(0)

    text = _KEYWORD_ROMAN_RE.sub(_kw, text)

    # b) Số La Mã đứng đầu chuỗi ("I", "II.", "III - ...") -> chữ.
    m = _LEADING_ROMAN_RE.match(text)
    if m:
        word = _roman_word(m.group(1))
        if word:
            text = text[:m.start(1)] + word + text[m.end(1):]
    return text


# --------------------------------------------------------------------------- #
# 2) Chữ VIẾT HOA -> chữ thường                                                #
# --------------------------------------------------------------------------- #

def _is_caps_word(word: str) -> bool:
    letters = [c for c in word if c.isalpha()]
    return bool(letters) and all(c.isupper() for c in letters)


def _cap_first(s: str) -> str:
    for i, ch in enumerate(s):
        if ch.isalpha():
            return s[:i] + ch.upper() + s[i + 1:]
    return s


def _heading_roman(words, k) -> bool:
    """Từ thứ k có phải số La Mã đầu mục ("PHẦN II", "IV. ...") không. Chỉ nhận ở
    vị trí chắc chắn, vì nhiều âm tiết tiếng Việt viết hoa trùng số La Mã (DI, MI, XI, LI)."""
    core = words[k].strip("()[].:-—–,")
    if not re.fullmatch(r"[IVXLCDM]+", core) or roman_to_int(core) is None:
        return False
    if k > 0 and words[k - 1].lower().strip("()[].:-—–,") in _INDEX_WORDS:
        return True
    return k == 0 and (len(words) == 1 or core != words[k].strip("(["))


def fix_allcaps(text: str, min_run: int = 3, ratio: float = 0.6) -> str:
    """Hạ chữ VIẾT HOA về chữ thường để TTS đọc đúng tiếng Việt.

    - Cả câu/tiêu đề phần lớn viết hoa (>= `ratio` số từ, tối thiểu 2 từ):
      chuyển về dạng câu ("TÔI ĐÃ SỐNG." -> "Tôi đã sống.").
    - Trong câu thường: chỉ hạ chuỗi >= `min_run` từ viết hoa liền nhau
      (chữ nhấn mạnh). Từ viết tắt đứng riêng (NXB, TP.HCM, UNESCO) giữ nguyên.
    """
    words = text.split(" ")
    caps = [_is_caps_word(w) for w in words]
    roman = [_heading_roman(words, k) for k in range(len(words))]
    n_words = sum(any(c.isalpha() for c in w) for w in words)
    if sum(caps) >= 2 and sum(caps) >= ratio * n_words:
        return _cap_first(" ".join(w.lower() if c and not r else w
                                   for w, c, r in zip(words, caps, roman)))
    out, i = list(words), 0
    while i < len(words):
        j = i
        while j < len(words) and caps[j]:
            j += 1
        if j - i >= min_run:
            for k in range(i, j):
                if not roman[k]:
                    out[k] = words[k].lower()
            if i == 0 or re.search(r"[.!?…:\"“]$", words[i - 1]):
                out[i] = _cap_first(out[i])
        i = max(j, i + 1)
    return " ".join(out)


# --------------------------------------------------------------------------- #
# 3) Lọc ký tự lạ (chống "babble" của neural TTS)                              #
# --------------------------------------------------------------------------- #

# Dấu câu / ký hiệu được PHÉP giữ lại (đọc được hoặc ảnh hưởng ngắt nghỉ).
_ALLOWED_PUNCT = set(".,;:!?…\"'`“”‘’«»()[]-—–/%")
# Khoảng trắng gộp về một dấu cách.
_WS_RE = re.compile(r"\s+")
# Có ít nhất một chữ cái hoặc chữ số -> mới đáng đọc.
_HAS_ALNUM_RE = re.compile(r"[^\W_]", re.UNICODE)


def _strip_exotic(text: str) -> str:
    """Bỏ ký tự không phải chữ/số/khoảng trắng/dấu-câu-cho-phép.

    Chữ có dấu tiếng Việt (và mọi chữ cái Unicode) là `isalpha()` nên được giữ;
    emoji, mũi tên, ký hiệu toán, box-drawing, ký tự điều khiển... bị loại.
    """
    out = []
    for ch in text:
        if ch.isalpha() or ch.isdigit() or ch.isspace() or ch in _ALLOWED_PUNCT:
            out.append(ch)
        # ký tự khác: bỏ (thay bằng khoảng trắng để không dính chữ)
        else:
            out.append(" ")
    return "".join(out)


def is_speakable(text: str) -> bool:
    """Sau chuẩn hoá, còn ít nhất một chữ cái/chữ số -> mới nên đưa vào TTS."""
    return bool(_HAS_ALNUM_RE.search(text or ""))


def normalize_for_tts(text: str) -> str:
    """Chuẩn hoá một đơn vị văn bản trước khi tổng hợp giọng.

    Trả về chuỗi đã: chuẩn hoá Unicode (NFC), hạ chữ VIẾT HOA về chữ thường,
    chuyển số La Mã ở đầu mục thành chữ tiếng Việt, loại ký tự lạ và gộp khoảng
    trắng. Có thể trả chuỗi rỗng
    (khi đầu vào chỉ gồm ký hiệu) — gọi kèm `is_speakable` để bỏ qua khi rỗng.
    """
    if not text:
        return ""
    text = unicodedata.normalize("NFC", text)
    text = _strip_exotic(text)
    text = _WS_RE.sub(" ", text).strip()
    text = fix_allcaps(text)
    text = _convert_roman_headings(text)
    return text


## 3b. Module tách câu (`pdf_units.py`)

Làm sạch text, tách câu và tạo khóa cache cho từng câu. Khóa cache tính trên **text sẽ đọc**, nên đổi quy tắc ở đây hoặc ở `text_norm.py` thì chỉ những câu bị ảnh hưởng phải render lại.

In [ ]:
%%writefile pdf_units.py
"""pdf_units.py - PDF -> danh sách "unit" (câu) để đọc bằng TTS.

Mỗi unit gồm: i (thứ tự), p (số thứ tự đoạn), ch (chương), page, hash, text, say.
`text` là nguyên văn (để xem lại), `say` là bản đã chuẩn hoá (text_norm.py) để đưa vào TTS.
`hash` = khóa cache của `say`, chỉ dùng lúc render. Đổi quy tắc làm sạch / tách câu / chuẩn hoá
thì hash của câu bị ảnh hưởng đổi theo, và chỉ những câu đó phải render lại.
"""
import hashlib
import os
import re
import unicodedata
from collections import Counter

try:
    import pymupdf as fitz
except ImportError:  # PyMuPDF bản cũ
    import fitz

from text_norm import normalize_for_tts, is_speakable

RULES_VERSION = "pdfu-3"

# --------------------------------------------------------------------------
# Khóa cache: sha1 của câu. Cùng một câu (cùng giọng) chỉ phải render một lần.
# --------------------------------------------------------------------------
def unit_hash(text: str) -> str:
    return hashlib.sha1(text.encode("utf-8")).hexdigest()[:16]


# --------------------------------------------------------------------------
# Làm sạch text
# --------------------------------------------------------------------------
_LIGATURES = {"\ufb00": "ff", "\ufb01": "fi", "\ufb02": "fl", "\ufb03": "ffi", "\ufb04": "ffl"}
_URL = re.compile(r"(https?://|www\.)\S+", re.I)
_CTRL = re.compile(r"[\u0000-\u0008\u000b\u000c\u000e-\u001f\u007f\u00ad\u200b-\u200f\u2028\u2029\ufeff]")
_WS = re.compile(r"[ \t\u00a0\u2009\u202f]+")
_BULLET = re.compile(r"^[\u2022\u25cf\u25aa\u25e6\u2023\u2043\*\-–—]+\s+")
_TERMINAL = re.compile(r"[.!?…:;][\"”’')\]]*$")


def clean_text(s: str) -> str:
    # NFC rất quan trọng với tiếng Việt: PDF hay trả về dạng tổ hợp (NFD)
    s = unicodedata.normalize("NFC", s)
    for k, v in _LIGATURES.items():
        s = s.replace(k, v)
    s = _CTRL.sub("", s)
    s = _URL.sub("", s)
    s = _WS.sub(" ", s)
    return s.strip()


def _has_speakable(s: str) -> bool:
    return any(ch.isalnum() for ch in s)


# --------------------------------------------------------------------------
# Trích text từ PDF
# --------------------------------------------------------------------------
def _ocr_blocks(page, lang, dpi):
    """OCR 1 trang bằng tesseract CLI (KHÔNG dùng get_textpage_ocr của PyMuPDF: nó làm
    mất khoảng trắng sau chữ có dấu tiếng Việt - đã thử, 'Hồ Chí' thành 'HồChí').
    Không có tọa độ thật, nên đoạn đầu/cuối trang được gán vị trí lề để còn lọc header/footer."""
    import shutil
    import subprocess
    if not shutil.which("tesseract"):
        raise RuntimeError("Chưa cài tesseract: apt-get install -y tesseract-ocr tesseract-ocr-vie")
    png = page.get_pixmap(dpi=dpi).tobytes("png")
    r = subprocess.run(["tesseract", "stdin", "stdout", "-l", lang, "--psm", "3"],
                       input=png, capture_output=True)
    text = r.stdout.decode("utf-8", errors="ignore")
    chunks = [c.strip() for c in re.split(r"\n\s*\n", text) if c.strip()]
    h = page.rect.height
    out = []
    for k, c in enumerate(chunks):
        if k == 0:
            out.append((0.0, h * 0.05, c))
        elif k == len(chunks) - 1:
            out.append((h * 0.95, h, c))
        else:
            out.append((h * 0.3, h * 0.7, c))
    return out


def _page_blocks(page, ocr, ocr_lang, ocr_dpi=300):
    """Trả về list (y0, y1, text) của các block văn bản, theo thứ tự đọc của PDF."""
    if ocr and len(page.get_text("text").strip()) < 30:
        return _ocr_blocks(page, ocr_lang, ocr_dpi)
    raw = page.get_text("blocks")
    out = []
    for b in raw:
        if len(b) >= 7 and b[6] != 0:  # bỏ block ảnh
            continue
        txt = b[4]
        if txt and txt.strip():
            out.append((b[1], b[3], txt))
    return out


def _sig(text):
    """Chữ ký để phát hiện header/footer lặp: số -> '#', chữ thường, gọn khoảng trắng."""
    return re.sub(r"\d+", "#", _WS.sub(" ", text.strip().lower()))


def extract_paragraphs(path, ocr=False, ocr_lang="vie", dehyphen="keep", margin=0.07, ocr_dpi=300):
    """-> (paragraphs, n_pages, info). paragraphs = list[dict(text, page)].

    dehyphen: 'keep' giữ dấu '-' ở cuối dòng (an toàn với tiếng Việt, vì tiếng Việt
              không ngắt từ giữa âm tiết); 'join' nối liền (hợp với sách tiếng Anh).
    margin:   phần trăm chiều cao trang ở trên/dưới coi là vùng header/footer.
    """
    doc = fitz.open(path)
    n = len(doc)
    pages = []
    for pno in range(n):
        page = doc[pno]
        h = page.rect.height
        blocks = _page_blocks(page, ocr, ocr_lang, ocr_dpi)
        pages.append((pno, h, blocks))

    # phát hiện header/footer: block nằm ở vùng lề và lặp trên >= 30% số trang (tối thiểu 3)
    cnt = Counter()
    for pno, h, blocks in pages:
        seen = set()
        for y0, y1, t in blocks:
            if y1 < h * margin * 2 or y0 > h * (1 - margin * 2):
                seen.add(_sig(t))
        cnt.update(seen)
    min_rep = max(3, int(0.3 * n))
    repeated = {s for s, c in cnt.items() if c >= min_rep and len(s) < 120}

    dropped = 0
    paras = []
    for pno, h, blocks in pages:
        for y0, y1, t in blocks:
            in_margin = y1 < h * margin * 2 or y0 > h * (1 - margin * 2)
            if in_margin and _sig(t) in repeated:
                dropped += 1
                continue
            if in_margin and re.fullmatch(r"\s*[\divxlcdm]{1,6}\s*", t.strip().lower()):
                dropped += 1  # số trang trơ trọi
                continue
            t = t.replace("\r", "")
            t = re.sub(r"-\n", "-" if dehyphen == "keep" else "", t)
            t = t.replace("\n", " ")
            t = clean_text(t)
            t = _BULLET.sub("", t)
            if not _has_speakable(t):
                continue
            # đoạn văn bị ngắt giữa chừng (sang block/trang mới): nối lại
            if paras and not _TERMINAL.search(paras[-1]["text"]) and t[:1].islower():
                paras[-1]["text"] += " " + t
            else:
                paras.append({"text": t, "page": pno + 1})
    info = {"dropped_blocks": dropped, "repeated_patterns": len(repeated)}
    return paras, n, info


def extract_chapters(path, n_pages, fallback_every=20):
    """Chương theo mục lục PDF (cấp 1). Không có TOC -> chia mỗi `fallback_every` trang.
    Lưu ý: mốc chương chỉ chính xác đến mức TRANG."""
    doc = fitz.open(path)
    toc = [t for t in doc.get_toc() if t[0] == 1 and 1 <= t[2] <= n_pages]
    chapters = []
    if toc:
        if toc[0][2] > 1:
            chapters.append({"title": "(Mở đầu)", "start_page": 1})
        for _, title, pg in toc:
            if chapters and chapters[-1]["start_page"] == pg:
                chapters[-1]["title"] += " / " + clean_text(title)
            else:
                chapters.append({"title": clean_text(title) or f"Chương tại trang {pg}", "start_page": pg})
        source = "toc"
    else:
        for i, pg in enumerate(range(1, n_pages + 1, fallback_every)):
            chapters.append({"title": f"Phần {i + 1} (trang {pg}+)", "start_page": pg})
        source = "fallback"
    for i, c in enumerate(chapters):
        c["id"] = i
    return chapters, source


# --------------------------------------------------------------------------
# Tách câu
# --------------------------------------------------------------------------
_ABBR = {
    "tp", "ts", "ths", "pgs", "gs", "bs", "ks", "cn", "tt", "tr", "nxb", "ql", "đ/c",
    "dr", "mr", "mrs", "ms", "prof", "st", "vs", "no", "etc", "v.v", "tp.hcm",
}
_BOUND = re.compile(r"([.!?…]+[\"”’')\]]*)\s+")
_OPENERS = "\"“‘'([–—-"


def split_sentences(par: str):
    res, start = [], 0
    for m in _BOUND.finditer(par):
        nxt = par[m.end():m.end() + 1]
        if not nxt:
            continue
        if not (nxt.isupper() or nxt.isdigit() or nxt in _OPENERS):
            continue
        term = m.group(1)
        if term == ".":
            words = par[start:m.start(1)].split()
            w = words[-1].strip("\"“‘'([").lower() if words else ""
            if w in _ABBR or (len(w) == 1 and w.isalpha()) or (w.isdigit() and len(w) <= 2):
                continue
        res.append(par[start:m.end(1)].strip())
        start = m.end()
    tail = par[start:].strip()
    if tail:
        res.append(tail)
    return res


def split_long(s: str, max_chars: int):
    """Cắt câu quá dài ở ; : – , gần max_chars nhất, không thì ở khoảng trắng."""
    out = []
    while len(s) > max_chars:
        window = s[:max_chars + 1]
        cut = -1
        for sep in ("; ", ": ", " – ", " — ", ", ", " "):
            idx = window.rfind(sep, int(max_chars * 0.4))
            if idx != -1:
                cut = idx + len(sep.rstrip()) if sep != " " else idx
                break
        if cut <= 0:
            cut = max_chars
        out.append(s[:cut].strip())
        s = s[cut:].strip()
    if s:
        out.append(s)
    return out


def _merge_short(parts, min_chars):
    """Gộp mảnh quá ngắn vào mảnh kề (trong cùng đoạn)."""
    if len(parts) <= 1:
        return parts
    res = []
    for p in parts:
        if res and len(p) < min_chars:
            res[-1] += " " + p
        else:
            res.append(p)
    if len(res) > 1 and len(res[0]) < min_chars:
        res[1] = res[0] + " " + res[1]
        res = res[1:]
    return res


# --------------------------------------------------------------------------
# Hàm chính
# --------------------------------------------------------------------------
def build_units(path, max_chars=220, min_chars=10, ocr=False, ocr_lang="vie",
                dehyphen="keep", chapter_fallback_pages=20, ocr_dpi=300):
    paras, n_pages, info = extract_paragraphs(path, ocr=ocr, ocr_lang=ocr_lang, dehyphen=dehyphen, ocr_dpi=ocr_dpi)
    chapters, chap_source = extract_chapters(path, n_pages, chapter_fallback_pages)

    def chapter_of(page):
        cid = 0
        for c in chapters:
            if c["start_page"] <= page:
                cid = c["id"]
        return cid

    units = []
    total_chars = sum(len(p["text"]) for p in paras)
    for pi, p in enumerate(paras):
        parts = []
        for s in split_sentences(p["text"]):
            parts.extend(split_long(s, max_chars))
        parts = _merge_short(parts, min_chars)
        for s in parts:
            say = normalize_for_tts(s)
            if not is_speakable(say):
                continue
            units.append({
                "i": len(units),
                "p": pi,
                "ch": chapter_of(p["page"]),
                "page": p["page"],
                "hash": unit_hash(say),
                "text": s,
                "say": say,
            })

    avg = total_chars / max(n_pages, 1)
    warnings = []
    if avg < 80:
        warnings.append(
            f"Trung bình chỉ {avg:.0f} ký tự/trang: nhiều khả năng PDF là bản SCAN (không có text). "
            "Cài tesseract rồi chạy lại với OCR=True."
        )
    stats = {
        "pages": n_pages, "paragraphs": len(paras), "units": len(units),
        "unique_units": len({u['hash'] for u in units}), "chars": total_chars,
        "chapter_source": chap_source, "rules": RULES_VERSION, **info,
    }
    return {"units": units, "chapters": chapters, "stats": stats, "warnings": warnings}

## 3c. Module sắc thái giọng đọc (`sac_thai.py`)

OmniVoice không có nút chỉnh cảm xúc: nó **bắt chước cách đọc của clip mẫu**. Vì vậy mỗi sắc thái gồm:
- một clip mẫu riêng của **cùng giọng đó**, đọc đúng sắc thái ấy (xem `huong-dan-thu-clip-mau.md` trong repo);
- tốc độ nói (`speed`) và độ dài khoảng nghỉ sau câu (`pause`).

Sắc thái nào chưa có clip thì dùng clip dự phòng (cuối cùng là giọng gốc ở mục 2b), nhưng **tốc độ và khoảng nghỉ vẫn đổi theo sắc thái**. Muốn chỉnh các thông số này, sửa `REGISTERS` trong ô dưới.

In [ ]:
%%writefile sac_thai.py
"""sac_thai.py - gán sắc thái giọng đọc cho từng câu dựa vào ngữ cảnh đoạn văn.

OmniVoice không có tham số "cảm xúc": model bắt chước cách đọc của clip mẫu. Vì vậy mỗi
sắc thái = (một clip mẫu cùng giọng, đọc đúng sắc thái đó) + tốc độ + độ dài khoảng nghỉ.
Thiếu clip của sắc thái nào thì dùng clip dự phòng (cuối cùng là giọng gốc), nhưng
tốc độ và khoảng nghỉ của sắc thái đó vẫn được áp dụng.

Thứ tự quyết định sắc thái của một câu:
    1. OVERRIDES thủ công {số thứ tự câu: sắc thái}
    2. Cache nhãn đã gán (tags.json trên Drive, không gọi lại LLM)
    3. Một LLM đọc đoạn văn kèm vài đoạn trước đó làm ngữ cảnh: LocalTagger (model mở chạy
       trên GPU Colab, miễn phí) hoặc ClaudeTagger (Claude API, tốn phí)
    4. Heuristic: câu là lời thoại (trong ngoặc kép / bắt đầu bằng gạch) -> doi_thoai
    5. trung_tinh
"""
import hashlib
import json
import re

# speed: hệ số tốc độ nói của OmniVoice (<1 chậm hơn). pause: nhân khoảng nghỉ sau câu.
REGISTERS = {
    "trung_tinh": dict(label="Kể chuyện, bình thản", speed=1.0, pause=1.0, fallback=()),
    "doi_thoai":  dict(label="Lời thoại nhân vật, sinh động", speed=1.0, pause=0.9,
                       fallback=("trung_tinh",)),
    "tinh_cam":   dict(label="Xúc động, buồn, thương, hoài niệm", speed=0.92, pause=1.25,
                       fallback=("trung_tinh",)),
    "kich_tinh":  dict(label="Căng thẳng, hồi hộp, sợ hãi, giận dữ", speed=1.06, pause=0.85,
                       fallback=("doi_thoai", "trung_tinh")),
    "vui_tuoi":   dict(label="Vui, hài hước, tươi tắn", speed=1.04, pause=0.9,
                       fallback=("doi_thoai", "trung_tinh")),
    "goi_cam":    dict(label="Thân mật, thì thầm, gợi cảm", speed=0.9, pause=1.35,
                       fallback=("tinh_cam", "trung_tinh")),
}
DEFAULT = "trung_tinh"


def _h(s):
    return hashlib.sha1(s.encode("utf-8")).hexdigest()[:16]


def resolve(reg, clips):
    """Sắc thái -> tên clip sẽ dùng (theo chuỗi dự phòng). `clips` = các sắc thái đã có clip."""
    for name in (reg,) + REGISTERS[reg]["fallback"] + (DEFAULT,):
        if name in clips:
            return name
    return DEFAULT


def unit_key(say, voice, speed):
    """Khóa cache audio. Giọng gốc ở tốc độ 1.0 giữ đúng khóa cũ (hash của text),
    nên các câu đã render trước khi có tính năng sắc thái vẫn dùng lại được."""
    if voice == DEFAULT and speed == 1.0:
        return _h(say)
    return _h(f"{voice}|{speed:g}|{say}")


# --------------------------------------------------------------------------
# Heuristic (miễn phí)
# --------------------------------------------------------------------------
_QUOTE = re.compile(r"[\"“”«»].+?[\"“”«»]", re.S)
_DASH = re.compile(r"^\s*[—–-]\s*\S")


def heuristic(text):
    t = text.strip()
    if _DASH.match(t):
        return "doi_thoai"
    quoted = "".join(_QUOTE.findall(t))
    if quoted and len(quoted) >= 0.5 * len(t):
        return "doi_thoai"
    return None


# --------------------------------------------------------------------------
# Claude
# --------------------------------------------------------------------------
_SYSTEM = """Bạn là đạo diễn lồng tiếng cho sách nói tiếng Việt. Mặc định mọi câu được đọc bằng \
giọng kể chuyện bình thản (trung_tinh). Nhiệm vụ của bạn: tìm những câu cần đọc bằng sắc thái KHÁC, \
dựa vào nội dung câu VÀ ngữ cảnh của đoạn văn (ai đang nói, cảnh đang diễn ra, cảm xúc của \
nhân vật/người kể).

Các sắc thái khác:
{regs}

Nguyên tắc:
- Chỉ liệt kê câu cần sắc thái khác; câu không liệt kê sẽ đọc trung_tinh. Không giải thích.
- Phần lớn sách là lời kể: chỉ chọn sắc thái khác khi cảm xúc trong câu rõ ràng.
- Giữ sắc thái ổn định trong một đoạn; chỉ đổi khi nội dung thật sự chuyển (vd lời kể -> lời \
thoại, hoặc cảm xúc dâng lên/lắng xuống). Đổi giọng liên tục từng câu nghe rất giả.
- Lời thoại mang cảm xúc mạnh thì chọn sắc thái cảm xúc đó thay vì doi_thoai.
- Tiêu đề, mục lục, chú thích: để trung_tinh."""

# USD / triệu token (input, output), theo trang Pricing ngày 2026-10-27. Kiểm tra lại khi giá đổi:
# https://platform.claude.com/docs/en/about-claude/pricing
PRICES = {
    "claude-opus-5-5": (4.00, 20.00),
    "claude-sonnet-5-5": (2.00, 10.00),
    "claude-haiku-5-5": (0.10, 0.50),   # prompt <= 100k token (mỗi lô ở đây nhỏ hơn nhiều)
}
CACHE_READ = {"claude-opus-5-5": 0.05, "claude-sonnet-5-5": 0.05, "claude-haiku-5-5": 0.10}


def _batches(units, size):
    """Chia theo ranh giới đoạn, mỗi lô khoảng `size` câu."""
    batch, cur_p = [], None
    for u in units:
        if batch and len(batch) >= size and u["p"] != cur_p:
            yield batch
            batch = []
        batch.append(u)
        cur_p = u["p"]
    if batch:
        yield batch


def build_prompt(batch, para_text, context_paras):
    """Các câu cần xét (đánh số, dòng trống = sang đoạn mới) kèm vài đoạn ngay trước làm ngữ cảnh."""
    first_p = batch[0]["p"]
    ctx = [para_text[p] for p in range(max(0, first_p - context_paras), first_p) if p in para_text]
    lines, cur_p = [], None
    for u in batch:
        if u["p"] != cur_p:
            lines.append("")
            cur_p = u["p"]
        lines.append(f"[{u['i']}] {u['text']}")
    return ("NGỮ CẢNH (các đoạn ngay trước, KHÔNG gán nhãn):\n"
            + ("\n\n".join(ctx) if ctx else "(đầu sách)")
            + "\n\nCÁC CÂU CẦN XÉT (dòng trống = sang đoạn mới):\n"
            + "\n".join(lines).strip())


class Usage:
    """Cộng dồn token thật từ response.usage và quy ra USD."""

    def __init__(self, model, batch):
        self.model, self.batch = model, batch
        self.inp = self.cache_w = self.cache_r = self.out = self.requests = 0

    def add(self, u):
        self.requests += 1
        self.inp += u.input_tokens or 0
        self.cache_w += getattr(u, "cache_creation_input_tokens", 0) or 0
        self.cache_r += getattr(u, "cache_read_input_tokens", 0) or 0
        self.out += u.output_tokens or 0

    def usd(self):
        if self.model not in PRICES:
            return None
        pi, po = PRICES[self.model]
        cost = (self.inp * pi + self.cache_w * pi * 1.25
                + self.cache_r * pi * CACHE_READ[self.model] + self.out * po) / 1e6
        return cost * (0.5 if self.batch else 1.0)

    def __str__(self):
        usd = self.usd()
        return (f"{self.requests} request | input {self.inp:,} | output {self.out:,} (gồm suy nghĩ) token"
                + (f" | ≈ ${usd:.3f}" + (" (đã giảm 50% batch)" if self.batch else "") if usd is not None else ""))


class ClaudeTagger:
    """Gán nhãn theo lô: mỗi lần gửi một nhóm đoạn văn, kèm vài đoạn trước đó làm ngữ cảnh.

    use_batch=True: gửi tất cả lô qua Message Batches API (rẻ hơn 50%, thường xong trong vài
    phút tới 1 giờ, tối đa 24 giờ). Mã batch được lưu vào state_path, nên tắt ô/runtime chết
    rồi chạy lại vẫn lấy được kết quả, không gửi (và trả tiền) lần hai.
    """

    def __init__(self, api_key, model="claude-opus-5-5", effort="low", batch_units=150,
                 context_paras=3, use_batch=True, state_path=None, poll_s=30, client=None):
        if client is None:
            import anthropic
            client = anthropic.Anthropic(api_key=api_key)
        self.client = client
        self.model = model
        self.effort = effort
        self.batch_units = batch_units
        self.context_paras = context_paras
        self.use_batch = use_batch
        self.state_path = state_path
        self.poll_s = poll_s
        self.usage = Usage(model, use_batch)
        others = [k for k in REGISTERS if k != DEFAULT]
        self.system = _SYSTEM.format(regs="\n".join(f"- {k}: {REGISTERS[k]['label']}" for k in others))
        self.schema = {
            "type": "object",
            "properties": {"labels": {"type": "array", "items": {
                "type": "object",
                "properties": {"i": {"type": "integer"}, "reg": {"type": "string", "enum": others}},
                "required": ["i", "reg"], "additionalProperties": False}}},
            "required": ["labels"], "additionalProperties": False,
        }

    def _batches(self, units):
        return _batches(units, self.batch_units)

    def _params(self, batch, para_text):
        prompt = build_prompt(batch, para_text, self.context_paras)
        return dict(model=self.model, max_tokens=16000, system=self.system,
                    output_config={"effort": self.effort,
                                   "format": {"type": "json_schema", "schema": self.schema}},
                    messages=[{"role": "user", "content": prompt}])

    def _parse(self, msg, ids):
        """Câu không được liệt kê -> trung_tinh. Trả về nhãn cho MỌI câu trong lô."""
        self.usage.add(msg.usage)
        if msg.stop_reason != "end_turn":
            raise RuntimeError(f"stop_reason={msg.stop_reason}")
        text = next(b.text for b in msg.content if b.type == "text")
        got = {i: DEFAULT for i in ids}
        got.update({d["i"]: d["reg"] for d in json.loads(text)["labels"] if d["i"] in got})
        return got

    def tag(self, units, para_text, on_batch=None, log=print):
        batches = list(self._batches(units))
        if self.use_batch:
            return self._tag_batch_api(batches, para_text, on_batch, log)
        out = {}
        for n, b in enumerate(batches, 1):
            try:
                msg = self.client.beta.messages.create(
                    **self._params(b, para_text),
                    betas=["server-side-fallback-2026-07-01"], fallbacks="default")
                got = self._parse(msg, [u["i"] for u in b])
            except Exception as e:  # lỗi một lô không làm hỏng cả lượt: lô đó dùng heuristic
                log(f"  lô {n}/{len(batches)} lỗi ({str(e)[:120]}), dùng heuristic")
                continue
            out.update(got)
            if on_batch:
                on_batch(got)
            log(f"  lô {n}/{len(batches)}: {sum(r != DEFAULT for r in got.values())}/{len(b)} câu có sắc thái riêng")
        return out

    def _tag_batch_api(self, batches, para_text, on_batch, log):
        import time
        from pathlib import Path
        state = None
        sp = Path(self.state_path) if self.state_path else None
        if sp and sp.exists():
            state = json.loads(sp.read_text(encoding="utf-8"))
            log(f"Tiếp tục batch đã gửi trước đó: {state['batch_id']}")
        if state is None:
            ids = {f"lo{n:04d}": [u["i"] for u in b] for n, b in enumerate(batches)}
            reqs = [{"custom_id": cid, "params": self._params(b, para_text)}
                    for cid, b in zip(ids, batches)]
            mb = self.client.messages.batches.create(requests=reqs)
            state = {"batch_id": mb.id, "ids": ids}
            if sp:
                sp.write_text(json.dumps(state), encoding="utf-8")
            log(f"Đã gửi {len(reqs)} lô qua Batch API ({mb.id}). Chờ kết quả... "
                "(có thể dừng ô này, lần chạy sau tự lấy tiếp)")
        while True:
            mb = self.client.messages.batches.retrieve(state["batch_id"])
            if mb.processing_status == "ended":
                break
            c = mb.request_counts
            log(f"  đang xử lý: xong {c.succeeded + c.errored}/{len(state['ids'])} lô")
            time.sleep(self.poll_s)
        out = {}
        for r in self.client.messages.batches.results(state["batch_id"]):
            ids = state["ids"].get(r.custom_id)
            if ids is None:
                continue
            if r.result.type != "succeeded":
                log(f"  {r.custom_id}: {r.result.type}, dùng heuristic")
                continue
            try:
                got = self._parse(r.result.message, ids)
            except Exception as e:
                log(f"  {r.custom_id} lỗi ({str(e)[:120]}), dùng heuristic")
                continue
            out.update(got)
            if on_batch:
                on_batch(got)
        if sp:
            sp.unlink(missing_ok=True)
        log(f"  xong: {len(out)} câu có nhãn từ Claude")
        return out


# --------------------------------------------------------------------------
# LLM chạy ngay trên GPU Colab (miễn phí API)
# --------------------------------------------------------------------------
_LOCAL_FORMAT = """

Định dạng trả lời: mỗi câu cần sắc thái khác ghi MỘT dòng "số: nhãn". Không viết gì khác. \
Nếu không có câu nào cần sắc thái khác, trả lời đúng một chữ: KHONG

Ví dụ trả lời:
12: doi_thoai
13: doi_thoai
27: tinh_cam"""

_LINE = re.compile(r"\[?(\d+)\]?\s*[:=\-–]\s*([a-z_]+)")


class LocalTagger:
    """Gán sắc thái bằng một LLM mở chạy trên GPU của Colab: không tốn tiền API.

    Dùng chung prompt ngữ cảnh với ClaudeTagger, nhưng trả lời dạng dòng "số: nhãn" (model nhỏ
    hay viết JSON hỏng). Nhãn lạ / số câu không có trong lô bị bỏ qua. Nhiều lô được sinh song
    song (gen_batch) cho nhanh. Gọi close() sau khi xong để trả VRAM cho OmniVoice.
    """

    def __init__(self, model_id="Qwen/Qwen3-4B-Instruct-2507", batch_units=40, context_paras=3,
                 gen_batch=4, max_new_tokens=400, model=None, tokenizer=None):
        if model is None:
            import torch
            from transformers import AutoModelForCausalLM, AutoTokenizer
            tokenizer = AutoTokenizer.from_pretrained(model_id)
            model = AutoModelForCausalLM.from_pretrained(model_id, torch_dtype=torch.float16,
                                                         device_map="cuda")
        tokenizer.padding_side = "left"           # sinh theo lô cần đệm bên trái
        if tokenizer.pad_token is None:
            tokenizer.pad_token = tokenizer.eos_token
        self.model, self.tok = model, tokenizer
        self.model_id = model_id
        self.batch_units = batch_units
        self.context_paras = context_paras
        self.gen_batch = gen_batch
        self.max_new_tokens = max_new_tokens
        self.allowed = {k for k in REGISTERS if k != DEFAULT}
        others = [k for k in REGISTERS if k != DEFAULT]
        self.system = (_SYSTEM.format(regs="\n".join(f"- {k}: {REGISTERS[k]['label']}" for k in others))
                       + _LOCAL_FORMAT)

    def _chat(self, prompt):
        msgs = [{"role": "system", "content": self.system}, {"role": "user", "content": prompt}]
        # enable_thinking=False: tắt chế độ suy nghĩ nếu dùng bản Qwen3 có chế độ này
        return self.tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True,
                                            enable_thinking=False)

    def _generate(self, prompts):
        import torch
        enc = self.tok([self._chat(p) for p in prompts], return_tensors="pt", padding=True,
                       add_special_tokens=False).to(self.model.device)
        with torch.no_grad():
            out = self.model.generate(**enc, max_new_tokens=self.max_new_tokens, do_sample=False,
                                      pad_token_id=self.tok.pad_token_id)
        return self.tok.batch_decode(out[:, enc["input_ids"].shape[1]:], skip_special_tokens=True)

    def parse(self, text, ids):
        got = {i: DEFAULT for i in ids}
        for m in _LINE.finditer(text):
            i, reg = int(m.group(1)), m.group(2)
            if i in got and reg in self.allowed:
                got[i] = reg
        return got

    def tag(self, units, para_text, on_batch=None, log=print):
        import time
        batches = list(_batches(units, self.batch_units))
        out, t0 = {}, time.time()
        for k in range(0, len(batches), self.gen_batch):
            group = batches[k:k + self.gen_batch]
            try:
                texts = self._generate([build_prompt(b, para_text, self.context_paras) for b in group])
            except Exception as e:  # vd hết VRAM: giảm gen_batch rồi chạy lại
                log(f"  lô {k + 1}-{k + len(group)} lỗi ({str(e)[:120]}), dùng heuristic")
                continue
            for b, text in zip(group, texts):
                got = self.parse(text, [u["i"] for u in b])
                out.update(got)
                if on_batch:
                    on_batch(got)
            done = k + len(group)
            log(f"  {done}/{len(batches)} lô, {time.time() - t0:.0f}s")
        return out

    def close(self):
        import gc
        import torch
        self.model = None
        gc.collect()
        torch.cuda.empty_cache()


# --------------------------------------------------------------------------
# Hàm chính
# --------------------------------------------------------------------------
def assign(units, clips, cache=None, overrides=None, tagger=None, save=None, log=print):
    """Gán u['reg'], u['voice'], u['speed'], u['key'] cho từng unit (sửa tại chỗ).

    cache: dict {hash của say: sắc thái} đọc/ghi lại tags.json. Trả về cache đã cập nhật.
    save:  hàm save(cache), gọi sau mỗi lô Claude trả về (ngắt giữa chừng không mất nhãn).
    """
    cache = dict(cache or {})
    overrides = overrides or {}
    todo = [u for u in units if u["i"] not in overrides and u["hash"] not in cache]
    if todo and tagger is not None:
        para_text = {}
        for u in units:
            para_text[u["p"]] = (para_text.get(u["p"], "") + " " + u["text"]).strip()
        hash_of = {u["i"]: u["hash"] for u in todo}

        def on_batch(got):
            cache.update({hash_of[i]: r for i, r in got.items()})
            if save:
                save(cache)

        log(f"Gán sắc thái cho {len(todo)} câu "
            f"(~{sum(len(u['text']) for u in todo) / 1000:.0f}k ký tự)...")
        tagger.tag(todo, para_text, on_batch=on_batch, log=log)
    for u in units:
        reg = overrides.get(u["i"]) or cache.get(u["hash"])
        if reg not in REGISTERS:
            reg = heuristic(u["text"]) or DEFAULT
        u["reg"] = reg
        u["voice"] = resolve(reg, clips)
        u["speed"] = REGISTERS[reg]["speed"]
        u["key"] = unit_key(u["say"], u["voice"], u["speed"])
    return cache


## 4. PDF → danh sách câu (xem kỹ trước khi render!)

**Đọc lướt phần đầu và phần giữa sách.** Đây là bước duy nhất ngăn bạn render 4 tiếng GPU cho một đống rác. Cần để ý:
- chú thích cuối trang, số trang, tiêu đề lặp bị đọc lẫn vào câu;
- sách hai cột hoặc có bảng thì thứ tự đọc có thể sai;
- chương: nếu PDF không có mục lục thì chia tạm mỗi 20 trang.

In [ ]:
import importlib, json
from collections import Counter
import text_norm, pdf_units
importlib.reload(text_norm)
importlib.reload(pdf_units)

OCR = False       # True nếu PDF là bản scan (chạy ô cài tesseract ngay bên dưới trước)
MAX_CHARS = 220   # câu dài hơn bị cắt ở dấu ; : , gần nhất

res = pdf_units.build_units(PDF, max_chars=MAX_CHARS, ocr=OCR)
units = res['units']
print(res['stats'])
for w in res['warnings']:
    print('⚠️', w)

cc = Counter(u['ch'] for u in units)
print(f"\nChương (nguồn: {res['stats']['chapter_source']}):")
for c in res['chapters'][:40]:
    print(f"  {c['id']:>3} tr.{c['start_page']:<4} {cc[c['id']]:>5} câu  {c['title'][:60]}")
if len(res['chapters']) > 40:
    print('  ...')

def show(title, rows):
    print(f'\n--- {title}')
    for u in rows:
        print(f"[{u['i']} | tr.{u['page']}] {u['text'][:160]}")

mid = len(units) // 2
show('8 câu đầu', units[:8])
show('8 câu ở giữa sách', units[mid:mid + 8])
show('3 câu dài nhất', sorted(units, key=lambda u: -len(u['text']))[:3])

changed = [u for u in units if u['say'] != u['text']]
print(f'\n--- {len(changed)} câu được chuẩn hoá trước khi đọc (nguyên văn → đọc thành), 15 câu đầu:')
for u in changed[:15]:
    print(f"[{u['i']} | tr.{u['page']}] {u['text'][:80]!r}\n{'':>12}→ {u['say'][:80]!r}")

with open(ROOT/'units.jsonl', 'w', encoding='utf-8') as f:
    for u in units:
        f.write(json.dumps(u, ensure_ascii=False) + '\n')
print('\nĐã lưu', ROOT/'units.jsonl')

**Chỉ chạy ô dưới nếu ô trên cảnh báo PDF là bản scan**, rồi quay lại đặt `OCR = True` và chạy lại ô trên.

OCR tiếng Việt vẫn nhận sai dấu (vd "rằng" thành "răng"), và TTS sẽ đọc đúng cái sai đó. Nếu có bản PDF có text layer thì luôn nên dùng bản đó.

In [ ]:
# !apt-get -qq install -y tesseract-ocr tesseract-ocr-vie > /dev/null
# print('tesseract sẵn sàng')

## 4b. Sắc thái giọng đọc theo ngữ cảnh

**Ô 1 (tùy chọn): upload clip sắc thái.** Đặt `UPLOAD_REGISTERS = True`, chọn một hoặc nhiều clip, **đặt tên file đúng tên sắc thái**: `doi_thoai.wav`, `tinh_cam.wav`, `kich_tinh.wav`, `vui_tuoi.wav`, `goi_cam.wav`. Clip mẫu ở mục 2b đóng vai `trung_tinh` (kể chuyện). Mỗi clip dài 6–10 giây, cùng một người đọc, cùng mic. Không upload clip nào cũng được: khi đó chỉ tốc độ và khoảng nghỉ thay đổi theo sắc thái.

**Ô 2: gán sắc thái cho từng câu.** Một LLM đọc từng đoạn văn kèm 3 đoạn trước đó để chọn sắc thái cho mỗi câu (ai đang nói, cảnh đang diễn ra, cảm xúc của nhân vật). Chọn bằng `TAGGER`:
- **`'local'` (mặc định, miễn phí):** model mở `Qwen3-4B-Instruct` chạy ngay trên GPU T4 của Colab. Lần đầu tải khoảng 8 GB, gán nhãn cả cuốn mất vài phút tới vài chục phút. Xong thì giải phóng VRAM cho OmniVoice. Model nhỏ hiểu cảm xúc tiếng Việt kém Claude, nên **xem kỹ các câu mẫu in ra** và nghe thử trước khi render cả cuốn. Bị hết VRAM thì giảm `LOCAL_GEN_BATCH`.
- **`'claude'` (tốn phí):** cần secret **`ANTHROPIC_API_KEY`** (biểu tượng 🔑 ở thanh bên trái → *Add new secret* → bật *Notebook access*). Mặc định gửi qua Batch API (rẻ hơn 50%, phải chờ vài phút tới 1 giờ). `TAG_MODEL = 'claude-haiku-5-5'` rẻ hơn Opus khoảng 40 lần. Cuối ô in số tiền thật của lượt chạy.
- **`'heuristic'`:** không dùng LLM. Câu là lời thoại thì đọc kiểu `doi_thoai`, còn lại đọc `trung_tinh`.

Nhãn được lưu vào `tags.json` trên Drive sau mỗi lô, nên chạy lại không phải gán lại. Muốn đổi cách gán thì đặt `RETAG = True`. Muốn sửa tay câu nào thì dùng `OVERRIDES` (số thứ tự câu in ở mục 4). Câu kể chuyện bình thường giữ nguyên khóa cache cũ, chỉ câu đổi sắc thái mới phải render lại.

In [ ]:
import re, shutil
from collections import Counter
from pathlib import Path
import importlib, sac_thai
importlib.reload(sac_thai)

UPLOAD_REGISTERS = False   # True = upload thêm clip sắc thái (tên file = tên sắc thái, vd tinh_cam.wav)

AUDIO_EXT = ('.wav', '.mp3', '.m4a', '.flac', '.ogg')
REG_DIR = ROOT/'registers'
REG_DIR.mkdir(exist_ok=True)
if UPLOAD_REGISTERS:
    up = files.upload()
    for n in up:
        stem = re.sub(r'\s*\(\d+\)$', '', Path(n).stem)   # Colab đổi 'x.wav' trùng tên thành 'x (1).wav'
        ext = Path(n).suffix.lower()
        if stem not in sac_thai.REGISTERS or stem == sac_thai.DEFAULT or ext not in AUDIO_EXT:
            print(f'⏭  bỏ qua {n}: tên file phải là một trong',
                  ', '.join(r for r in sac_thai.REGISTERS if r != sac_thai.DEFAULT))
            continue
        for old in REG_DIR.glob(stem + '.*'):
            old.unlink()
        shutil.copy(n, REG_DIR/(stem + ext))
        print('Đã lưu clip sắc thái:', stem + ext)

REG_CLIPS = {p.stem: str(p) for p in sorted(REG_DIR.iterdir())
             if p.stem in sac_thai.REGISTERS and p.suffix.lower() in AUDIO_EXT}
REG_CLIPS[sac_thai.DEFAULT] = REF_AUDIO
print(f"{'sắc thái':<11} {'clip dùng':<11} speed  nghỉ   mô tả")
for name, spec in sac_thai.REGISTERS.items():
    voice = sac_thai.resolve(name, REG_CLIPS)
    print(f"{name:<11} {voice + ('' if voice == name else ' ↩'):<11} {spec['speed']:<5}  {spec['pause']:<5}  {spec['label']}")
print('↩ = chưa có clip riêng, dùng clip dự phòng')

In [ ]:
import json
from collections import Counter

TAGGER = 'local'               # 'local' = LLM mở trên GPU Colab (miễn phí) | 'claude' = Claude API (tốn phí) | 'heuristic'
LOCAL_MODEL = 'Qwen/Qwen3-4B-Instruct-2507'
LOCAL_GEN_BATCH = 4            # số lô sinh song song; hết VRAM thì giảm xuống 2 hoặc 1
TAG_MODEL = 'claude-opus-5-5'  # chỉ dùng khi TAGGER = 'claude'; 'claude-haiku-5-5' rẻ hơn khoảng 40 lần
TAG_EFFORT = 'low'
USE_BATCH = True               # Claude: True = Batch API, rẻ hơn 50% nhưng phải chờ
OVERRIDES = {}                 # sửa tay: {số thứ tự câu: 'tinh_cam', ...}
RETAG = False                  # True = bỏ nhãn đã lưu, gán lại từ đầu

assert all('say' in u for u in units), 'Hãy chạy lại mục 4 (danh sách câu) trước.'
TAGS = ROOT/'tags.json'
tag_cache = {} if RETAG or not TAGS.exists() else json.loads(TAGS.read_text(encoding='utf-8'))
save_tags = lambda c: TAGS.write_text(json.dumps(c, ensure_ascii=False), encoding='utf-8')

tagger = None
todo = [u for u in units if u['i'] not in OVERRIDES and u['hash'] not in tag_cache]
if TAGGER == 'local' and todo:
    import torch
    assert torch.cuda.is_available(), 'TAGGER = "local" cần GPU (Runtime → Change runtime type → T4 GPU)'
    tagger = sac_thai.LocalTagger(LOCAL_MODEL, gen_batch=LOCAL_GEN_BATCH)
elif TAGGER == 'claude' and todo:
    from google.colab import userdata
    try:
        api_key = userdata.get('ANTHROPIC_API_KEY')
    except Exception:
        api_key = None
    if api_key:
        tagger = sac_thai.ClaudeTagger(api_key, model=TAG_MODEL, effort=TAG_EFFORT, use_batch=USE_BATCH,
                                       state_path=ROOT/'tag_batch.json')
    else:
        print('⚠️ Chưa có secret ANTHROPIC_API_KEY (hoặc chưa bật Notebook access) → dùng heuristic.')

tag_cache = sac_thai.assign(units, set(REG_CLIPS), cache=tag_cache, overrides=OVERRIDES,
                            tagger=tagger, save=save_tags)
save_tags(tag_cache)
if isinstance(tagger, sac_thai.LocalTagger):
    tagger.close()             # trả VRAM cho OmniVoice ở mục 6
elif tagger and tagger.usage.requests:
    print('Chi phí lượt này:', tagger.usage)

cnt = Counter(u['reg'] for u in units)
print('\nSố câu theo sắc thái:', dict(cnt.most_common()))
for name in sac_thai.REGISTERS:
    rows = [u for u in units if u['reg'] == name][:3]
    if rows and name != sac_thai.DEFAULT:
        print(f'\n--- {name} (đọc bằng clip {rows[0]["voice"]}, speed {rows[0]["speed"]})')
        for u in rows:
            print(f"[{u['i']} | tr.{u['page']}] {u['text'][:150]}")
print('\nChỗ nào chưa hợp lý: thêm vào OVERRIDES rồi chạy lại ô này (không tốn thêm phí).')

## 5. Cấu hình giọng clone

Điền **đúng lời trong clip** vào `REF_TEXT`. Để trống thì model nạp thêm Whisper để tự chép, mà chép sai một chữ là clone kém đi. Clip sắc thái ở mục 4b thì điền vào `REG_TEXTS`.

In [ ]:
REF_TEXT = ''        # lời ĐÚNG trong clip, vd: 'Xin chào, đây là giọng đọc thử của tôi.'
REG_TEXTS = {}       # lời đúng của từng clip sắc thái (mục 4b), vd {'tinh_cam': 'Mẹ ơi, con về rồi đây...'}
NUM_STEP = 32        # 16 = nhanh ~2x, 32 = nét hơn. Nên nghe thử cả hai trước khi render cả cuốn.
VOICE_ID = 'clone1'  # ĐỔI tên khi đổi clip giọng / REF_TEXT, để cache không bị lẫn
SHARD_SIZE = 300     # cứ 300 câu lưu 1 đợt lên Drive (runtime chết thì mất tối đa chừng này)
CPS_MIN, CPS_MAX = 6, 50   # ngưỡng QC (ký tự/giây); chỉnh lại theo số liệu benchmark

## 6. Nạp model + tạo prompt giọng clone cho từng clip sắc thái (làm 1 lần)

In [ ]:
import torch, numpy as np, soundfile as sf
from omnivoice import OmniVoice

model = OmniVoice.from_pretrained('k2-fsa/OmniVoice', device_map='cuda', dtype=torch.float16)

texts = {**REG_TEXTS, sac_thai.DEFAULT: REF_TEXT}
PROMPTS = {}
for name in sorted({u['voice'] for u in units} | {sac_thai.DEFAULT}):   # chỉ những clip thực sự được dùng
    kw = {'ref_audio': REG_CLIPS[name]}
    if texts.get(name, '').strip():
        kw['ref_text'] = texts[name].strip()
    PROMPTS[name] = model.create_voice_clone_prompt(**kw)
print('Đã có giọng clone:', ', '.join(PROMPTS))

## 7. Render

**Lần đầu để `LIMIT = 30`**: chỉ render 30 câu để đo tốc độ và ước tính tổng thời gian. Nghe thử vài file trong `cache/`. Nếu ETA hợp lý thì đặt `LIMIT = None` và chạy tiếp.

- Cứ `SHARD_SIZE` câu lại lưu một shard `.tar` lên Drive. Runtime chết thì mount lại, điền `BOOK`, chạy lại từ mục 2 đến mục 7, nó tự khôi phục và render tiếp phần thiếu.
- Mỗi câu được kiểm tra (không im lặng, không quá ngắn, tốc độ nói hợp lý). Câu lỗi thử lại 1 lần, vẫn lỗi thì ghi vào `failed.jsonl` và **không** vào cache.
- Giữ nguyên thứ tự sách, để các chương đầu hoàn tất trước.

In [ ]:
import json, os, shutil, tarfile, tempfile, time
from pathlib import Path

LIMIT = 30            # benchmark: render 30 câu rồi dừng. Đặt None để render hết.
RETRY_FAILED = False  # True = thử lại cả những câu bị loại ở lần chạy trước

OUT = Path('cache')/f'{VOICE_ID}-s{NUM_STEP}'      # tách theo giọng/step, không bao giờ lẫn
OUT.mkdir(parents=True, exist_ok=True)
SHARDS = ROOT/f'{VOICE_ID}-s{NUM_STEP}'
SHARDS.mkdir(exist_ok=True)
FAILED, DONE = SHARDS/'failed.jsonl', SHARDS/'done.txt'
SR = getattr(model, 'sampling_rate', 24000)

# 1) Phiên mới (đĩa trống): khôi phục những gì đã render từ Drive
if not any(OUT.glob('*.wav')):
    shards = sorted(SHARDS.glob('shard_*.tar'))
    for tp in shards:
        with tarfile.open(tp) as tf:
            try:
                tf.extractall(OUT, filter='data')
            except TypeError:
                tf.extractall(OUT)
    print(f'Khôi phục {len(shards)} shard từ Drive → {len(list(OUT.glob("*.wav")))} file')

# 2) File có trên đĩa nhưng chưa nằm trong shard nào (kernel chết giữa chừng) → đưa vào hàng đợi lưu
done = set(DONE.read_text().split()) if DONE.exists() else set()
buf = sorted({p.stem for p in OUT.glob('*.wav')} - done)

def flush():
    """Gói các file mới thành 1 shard rồi đẩy lên Drive (ghi .part rồi đổi tên để không bao giờ có shard dở)."""
    global buf
    if not buf:
        return
    name = f'shard_{len(list(SHARDS.glob("shard_*.tar"))):04d}.tar'
    tmp = Path(tempfile.gettempdir())/name
    with tarfile.open(tmp, 'w') as tf:
        for h in buf:
            tf.add(OUT/f'{h}.wav', arcname=f'{h}.wav')
    part = SHARDS/(name + '.part')
    shutil.copy(tmp, part)
    os.replace(part, SHARDS/name)
    tmp.unlink()
    with open(DONE, 'a') as f:
        f.write('\n'.join(buf) + '\n')
    print(f'  💾 đã lưu {name} ({len(buf)} file) lên Drive')
    buf = []

def to_np(a):
    if hasattr(a, 'detach'):
        a = a.detach().cpu().float().numpy()
    return np.squeeze(np.asarray(a, dtype=np.float32))

def qc(a, text):
    dur = len(a) / SR
    if dur < 0.3:
        return 'too_short', 0
    if float(np.max(np.abs(a))) < 0.01:
        return 'silent', 0
    cps = len(text) / dur
    if cps > CPS_MAX:
        return 'too_fast (cụt?)', cps
    if cps < CPS_MIN:
        return 'too_slow (dài/ảo?)', cps
    return 'ok', cps

# 3) Hàng đợi: bỏ trùng (giữ thứ tự sách), bỏ câu đã có, bỏ câu từng bị loại
failed = set()
if FAILED.exists():
    failed = {json.loads(l)['hash'] for l in open(FAILED, encoding='utf-8') if l.strip()}
uniq = {}
for u in units:
    uniq.setdefault(u['key'], (u['say'], u['voice'], u['speed']))
have = {p.stem for p in OUT.glob('*.wav')}
pending = [(h, *v) for h, v in uniq.items() if h not in have and (RETRY_FAILED or h not in failed)]
print(f'{len(units)} unit → {len(uniq)} câu khác nhau | đã có {len(have & set(uniq))} | '
      f'cần render {len(pending)} | từng bị loại {len(failed)}')

todo = pending[:LIMIT] if LIMIT else pending
ok = bad = 0
cps_list = []
t0 = time.time()
try:
    for i, (h, t, voice, speed) in enumerate(todo, 1):
        status, cps, a = 'error', 0, None
        for attempt in range(2):
            try:
                extra = {} if speed == 1.0 else {'speed': speed}
                a = to_np(model.generate(text=t, voice_clone_prompt=PROMPTS[voice], num_step=NUM_STEP, **extra)[0])
                status, cps = qc(a, t)
                if status == 'ok':
                    break
            except torch.cuda.OutOfMemoryError:
                torch.cuda.empty_cache()
                status = 'oom'
            except Exception as e:
                status = 'error: ' + str(e)[:100]
        if status == 'ok':
            sf.write(str(OUT/f'{h}.wav'), a, SR)
            buf.append(h)
            ok += 1
            cps_list.append(cps)
        else:
            bad += 1
            with open(FAILED, 'a', encoding='utf-8') as f:
                f.write(json.dumps({'hash': h, 'text': t, 'voice': voice, 'speed': speed, 'why': status},
                                   ensure_ascii=False) + '\n')
        if len(buf) >= SHARD_SIZE:
            flush()
        if i % 20 == 0 or i == len(todo):
            sp = i / (time.time() - t0)
            print(f'{i}/{len(todo)}  ok={ok} loại={bad}  ~{sp:.2f} câu/s  còn ~{(len(todo) - i) / sp / 3600:.1f} giờ')
finally:
    flush()   # kể cả khi bạn bấm Stop, phần đã render vẫn được lưu

elapsed = time.time() - t0
if cps_list:
    p = np.percentile(cps_list, [1, 5, 50, 95, 99])
    print('\nKý tự/giây của câu đạt (p1, p5, p50, p95, p99):', np.round(p, 1),
          '→ chỉnh CPS_MIN / CPS_MAX cho khớp rồi mới render cả cuốn')
if LIMIT and todo:
    rate = sum(len(t) for _, t, *_ in todo) / elapsed                      # ký tự/giây xử lý
    eta_h = sum(len(t) for _, t, *_ in pending) / rate / 3600
    print(f'\n⏱ Benchmark: ước tính render hết {len(pending)} câu cần ~{eta_h:.1f} giờ GPU '
          f'(Colab free thường ngắt trước ~10-12 giờ, hay sớm hơn).')
    print('   Quá lâu? Giảm NUM_STEP, hoặc cân nhắc GPU thuê theo giờ thay vì Colab free.')
elif not pending:
    print('\nKhông còn câu nào cần render.')
else:
    print('\nXONG lượt render.')

## 8. Báo cáo theo chương

Cho biết mỗi chương đã render được bao nhiêu câu. Câu nào bị loại (`failed.jsonl`) hoặc chưa render sẽ bị **bỏ qua** khi ghép mp3, nên chương nào chưa 100% sẽ có khoảng trống trong nội dung.

In [ ]:
have = {p.stem for p in OUT.glob('*.wav')}
tot, dn = Counter(), Counter()
for u in units:
    tot[u['ch']] += 1
    dn[u['ch']] += u['key'] in have

print('   chương  đã render     %')
for c in res['chapters']:
    n, d = tot[c['id']], dn[c['id']]
    mark = '✅' if n and d == n else '  '
    print(f"{mark} {c['id']:>3}   {d:>5}/{n:<5} {100 * d / max(n, 1):5.1f}%  {c['title'][:50]}")
regs = Counter((u['reg'], u['key'] in have) for u in units)
print('\nTheo sắc thái (đã render/tổng):',
      ', '.join(f"{r} {regs[(r, True)]}/{regs[(r, True)] + regs[(r, False)]}" for r in sac_thai.REGISTERS
                if regs[(r, True)] + regs[(r, False)]))
if FAILED.exists():
    print('\nCâu bị loại (xem lại / render bù bằng RETRY_FAILED=True):', FAILED)

## 9. Ghép thành mp3

Cần đã render xong (mục 7). Nếu runtime vừa khởi động lại, chạy lại mục 2 → 7 trước, mục 7 sẽ tự khôi phục từ Drive và báo không còn câu cần render.

- `MODE = 'chapter'`: mỗi chương một file, kèm thẻ ID3 (tên chương, số thứ tự) nên app nghe sách tự xếp đúng thứ tự. `MODE = 'book'`: cả cuốn một file.
- Mỗi câu được cân âm lượng về cùng mức, rồi nghỉ ngắn giữa câu, nghỉ lâu hơn khi hết đoạn.
- Chỉ cần ghép lại (đổi `MODE`, bitrate, độ nghỉ) thì **không phải render lại**.
- Chương đã ghép và không thay đổi (cùng các câu, cùng sắc thái, cùng độ nghỉ) sẽ được bỏ qua ở lần chạy sau.
- Khoảng nghỉ được nhân theo sắc thái của câu (vd `tinh_cam` nghỉ lâu hơn), chỉnh ở `REGISTERS` mục 3c.
- Dung lượng: khoảng **29 MB mỗi giờ audio** ở 64 kbps.

In [ ]:
import hashlib, json, os, re, shutil, subprocess, tempfile, unicodedata
import numpy as np, soundfile as sf
from pathlib import Path

MODE = 'chapter'        # 'chapter' = mỗi chương 1 file mp3 | 'book' = cả cuốn 1 file
BOOK_TITLE = ''         # tên hiển thị trong app nghe sách (trống = dùng tên thư mục)
MP3_KBPS = 64           # mono 64 kbps ≈ 29 MB/giờ; 48 nhỏ hơn, 96 nét hơn
PAUSE_SENT, PAUSE_CLAUSE, PAUSE_PARA = 0.35, 0.15, 0.8   # giây nghỉ: hết câu | hết mệnh đề bị cắt | hết đoạn
ALLOW_MISSING = True    # True = chương thiếu câu vẫn ghép (bỏ câu thiếu, liệt kê trong missing.txt); False = bỏ qua cả chương
REBUILD_MP3 = False     # True = ghép lại tất cả dù file đã có

assert shutil.which('ffmpeg'), 'Thiếu ffmpeg: chạy  !apt-get -qq install -y ffmpeg'
assert any(OUT.glob('*.wav')), 'Chưa có file render nào. Chạy mục 7 trước.'

MP3_DIR = ROOT/'mp3'
MP3_DIR.mkdir(exist_ok=True)
BUILT = MP3_DIR/'built.json'
built = json.loads(BUILT.read_text(encoding='utf-8')) if BUILT.exists() else {}
_END = re.compile(r"[.!?…][\"”’')\]]*$")

def fslug(s):
    s = s.replace('đ', 'd').replace('Đ', 'D')
    s = unicodedata.normalize('NFKD', s).encode('ascii', 'ignore').decode()
    return re.sub(r'[^a-zA-Z0-9]+', '_', s).strip('_').lower()[:40] or 'untitled'

def norm(a, target_db=-20.0):
    # đưa mỗi câu về cùng mức âm lượng (giới hạn ±6 dB, không để méo)
    rms = float(np.sqrt(np.mean(a ** 2))) + 1e-9
    a = a * float(np.clip(10 ** ((target_db - 20 * np.log10(rms)) / 20), 0.5, 2.0))
    peak = float(np.max(np.abs(a)))
    return a * min(1.0, 0.98 / peak) if peak > 0 else a

def pcm(a):
    return (np.clip(a, -1, 1) * 32767).astype('<i2').tobytes()

def gap_after(u, nxt):
    if nxt is None:
        return 0.0
    scale = sac_thai.REGISTERS[u.get('reg', sac_thai.DEFAULT)]['pause']   # vd tình cảm nghỉ lâu hơn
    if nxt.get('p') != u.get('p'):
        return PAUSE_PARA * scale
    return (PAUSE_SENT if _END.search(u['text']) else PAUSE_CLAUSE) * scale

def encode(group, title, track, total, path):
    cmd = ['ffmpeg', '-y', '-loglevel', 'error', '-f', 's16le', '-ar', str(SR), '-ac', '1', '-i', 'pipe:0',
           '-codec:a', 'libmp3lame', '-b:a', f'{MP3_KBPS}k', '-id3v2_version', '3',
           '-metadata', f'title={title}', '-metadata', f'album={BOOK_TITLE or BOOK}', '-metadata', f'track={track}/{total}',
           str(path)]
    proc = subprocess.Popen(cmd, stdin=subprocess.PIPE)
    secs, missing = 0.0, []
    try:
        for k, u in enumerate(group):
            f = OUT/f"{u['key']}.wav"
            if not f.exists():
                missing.append(u)
                continue
            a, sr = sf.read(str(f), dtype='float32')
            assert sr == SR, f'{f.name}: sample rate {sr} khác {SR}'
            if a.ndim > 1:
                a = a.mean(axis=1)
            proc.stdin.write(pcm(norm(a)))
            gap = gap_after(u, group[k + 1] if k + 1 < len(group) else None)
            if gap:
                proc.stdin.write(np.zeros(int(SR * gap), dtype='<i2').tobytes())
            secs += len(a) / SR + gap
    finally:
        proc.stdin.close()
        rc = proc.wait()
    if rc != 0:
        raise RuntimeError('ffmpeg lỗi, xem thông báo phía trên')
    return secs, missing

by_ch = {}
for u in units:
    by_ch.setdefault(u['ch'], []).append(u)
if MODE == 'book':
    groups = [(BOOK_TITLE or BOOK, units)]
else:
    groups = [(c['title'], by_ch[c['id']]) for c in res['chapters'] if c['id'] in by_ch]
w = len(str(len(groups)))

active = []   # các nhóm hiện có file mp3 (vừa ghép hoặc đã có sẵn)
for n, (title, group) in enumerate(groups, 1):
    name = f'{BOOK}.mp3' if MODE == 'book' else f'{n:0{w}d}_{fslug(title)}.mp3'
    have_keys = [u['key'] for u in group if (OUT/f"{u['key']}.wav").exists()]
    present = len(have_keys)
    sig = hashlib.sha1('|'.join(have_keys + [f'{PAUSE_SENT},{PAUSE_CLAUSE},{PAUSE_PARA},{MP3_KBPS}']).encode()).hexdigest()[:16]
    if present == 0:
        print(f'⏭  {name}: chưa có câu nào được render, bỏ qua')
        continue
    if present < len(group) and not ALLOW_MISSING:
        print(f'⏭  {name}: thiếu {len(group) - present}/{len(group)} câu (ALLOW_MISSING=False), bỏ qua')
        continue
    if not REBUILD_MP3 and (MP3_DIR/name).exists() and built.get(name) == sig:
        print(f'✔  {name}: đã có, không đổi')
        active.append((title, group))
        continue
    tmp = Path(tempfile.gettempdir())/name
    secs, missing = encode(group, title, n, len(groups), tmp)
    part = MP3_DIR/(name + '.part')
    shutil.copy(tmp, part)
    os.replace(part, MP3_DIR/name)
    mb = tmp.stat().st_size / 1e6
    tmp.unlink()
    built[name] = sig
    BUILT.write_text(json.dumps(built, ensure_ascii=False, indent=1), encoding='utf-8')
    h, m, s_ = int(secs // 3600), int(secs % 3600 // 60), int(secs % 60)
    note = f'  ⚠️ thiếu {len(missing)} câu' if missing else ''
    print(f'🎧 {name}: {h}:{m:02d}:{s_:02d} | {mb:.1f} MB{note}')
    active.append((title, group))

missing_lines = [f"[{t[:40]} | tr.{u['page']}] {u['text']}"
                 for t, g in active for u in g if not (OUT/f"{u['key']}.wav").exists()]
MISSING = MP3_DIR/'missing.txt'
if missing_lines:
    MISSING.write_text('\n'.join(missing_lines), encoding='utf-8')
    print(f'\n⚠️ {len(missing_lines)} câu bị bỏ qua trong các file mp3 hiện có, danh sách: {MISSING}')
elif MISSING.exists():
    MISSING.unlink()
print(f'\nXong. Tải về từ Drive: omnivoice_render/{BOOK}/mp3/  (chọn thư mục → chuột phải → Download)')

## 10. Tải về

File mp3 nằm trên Drive tại `omnivoice_render/<sách>/mp3/`. Mở drive.google.com, chọn thư mục `mp3` → **Download** (Drive tự nén và tải về). Với file hàng trăm MB, cách này ổn định hơn nhiều so với tải thẳng từ Colab.

Muốn nghe thử nhanh một chương ngắn ngay trong Colab, bỏ dấu `#` ở dòng dưới:

In [ ]:
# files.download(str(sorted(MP3_DIR.glob('*.mp3'))[0]))